In [ ]:
# ──────────────────────────────────────────────────────────────
# 1. ENV SETUP
# ──────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv

load_dotenv(override=True)

llmgw_api_key = os.getenv("LLMGW_API_KEY")
llmgw_base_url = os.getenv("LLMGW_BASE_URL", "https://llmgw-wp.tekstac.com")

if not llmgw_api_key:
    raise ValueError("LLMGW_API_KEY not found. Add it to your .env file.")

# Set Anthropic-compatible env vars
os.environ["ANTHROPIC_API_KEY"] = llmgw_api_key
os.environ["ANTHROPIC_BASE_URL"] = llmgw_base_url

print("=" * 55)
print("ENVIRONMENT STATUS")
print("=" * 55)
print(f"LLMGW_API_KEY  : {'✅ set' if llmgw_api_key else '❌ missing'}")
print(f"LLMGW_BASE_URL : {llmgw_base_url}")
print("=" * 55)


# ──────────────────────────────────────────────────────────────
# 2. IMPORTS
# ──────────────────────────────────────────────────────────────
import operator
from typing import TypedDict, Annotated, Literal

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, ToolMessage, BaseMessage
from langgraph.graph import StateGraph, START, END


# ──────────────────────────────────────────────────────────────
# 3. MOCK DATABASE
# ──────────────────────────────────────────────────────────────
ORDER_DB = {
    "ORD123": {
        "status": "Shipped",
        "item": "Wireless Headphones",
        "expected_delivery": "Tomorrow"
    },
    "ORD456": {
        "status": "Processing",
        "item": "Mechanical Keyboard",
        "expected_delivery": "Next Tuesday"
    },
    "ORD789": {
        "status": "Delivered",
        "item": "Gaming Mouse",
        "expected_delivery": "Delivered Yesterday"
    },
    "ORD321": {
        "status": "Out for Delivery",
        "item": "Laptop Stand",
        "expected_delivery": "Today"
    },
    "ORD654": {
        "status": "Cancelled",
        "item": "Smart Watch",
        "expected_delivery": "N/A"
    },
    "ORD999": {
        "status": "Delayed",
        "item": "Bluetooth Speaker",
        "expected_delivery": "In 3 days"
    },
    "ORD777": {
        "status": "Processing",
        "item": "USB Hub",
        "expected_delivery": "Next Monday"
    },
    "ORD888": {
        "status": "Shipped",
        "item": "Laptop Bag",
        "expected_delivery": "Tomorrow"
    }
}


# ──────────────────────────────────────────────────────────────
# 4. MCP TOOL SCHEMAS (NO @tool DECORATOR)
# ──────────────────────────────────────────────────────────────
mcp_tools = [
    {
        "name": "check_order_status",
        "description": "Check the shipping status and expected delivery of a customer's order. Requires an order ID like ORD123.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID, for example ORD123"
                }
            },
            "required": ["order_id"]
        }
    },
    {
        "name": "calculate_refund_eligibility",
        "description": "Determine if a customer is eligible for a refund based on how many days ago they purchased the item.",
        "input_schema": {
            "type": "object",
            "properties": {
                "days_since_purchase": {
                    "type": "integer",
                    "description": "Number of days since the purchase was made"
                }
            },
            "required": ["days_since_purchase"]
        }
    }
]


# ──────────────────────────────────────────────────────────────
# 5. MCP TOOL EXECUTION LAYER
#    (This simulates what an MCP server would do)
# ──────────────────────────────────────────────────────────────
def mcp_execute(tool_name: str, args: dict) -> str:
    """
    Simulated MCP dispatcher:
    Receives tool name + args and executes the matching business logic.
    """

    print(f"[MCP] Executing tool: {tool_name} | args: {args}")

    if tool_name == "check_order_status":
        order_id = args["order_id"].upper()
        order = ORDER_DB.get(order_id)

        if order:
            return (
                f"Order {order_id}: {order['item']} is currently '{order['status']}'. "
                f"Expected delivery: {order['expected_delivery']}."
            )
        return f"I could not find order {order_id} in the system. Please verify the number."

    elif tool_name == "calculate_refund_eligibility":
        days_since_purchase = args["days_since_purchase"]

        if days_since_purchase <= 30:
            return "Eligible for a full refund."
        elif days_since_purchase <= 60:
            return "Eligible for store credit only."
        else:
            return "Not eligible for a refund. The 60-day return window has expired."

    return f"Unknown MCP tool: {tool_name}"


# ──────────────────────────────────────────────────────────────
# 6. MODEL SETUP
# ──────────────────────────────────────────────────────────────
chat_model = ChatAnthropic(
    model="global.anthropic.claude-sonnet-4-6",
    temperature=0.3,
    anthropic_api_key=llmgw_api_key,
    base_url=llmgw_base_url,
)

# Bind MCP-style tools directly to the model
llm_with_tools = chat_model.bind_tools(mcp_tools)

# Optional connection test
try:
    test = chat_model.invoke("Capital of France?")
    print(f"\n✅ Connection OK: {test.content}")
except Exception as e:
    print(f"❌ Connection Error: {e}")


# ──────────────────────────────────────────────────────────────
# 7. AGENT STATE
# ──────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]


# ──────────────────────────────────────────────────────────────
# 8. GRAPH NODES
# ──────────────────────────────────────────────────────────────
def support_agent_node(state: AgentState):
    """
    The LLM decides whether to:
    - answer directly
    - or call one/more MCP tools
    """
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


def tool_execution_node(state: AgentState):
    """
    Execute MCP tool calls requested by the LLM.
    """
    last_message = state["messages"][-1]
    results = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        # Execute via MCP dispatcher
        tool_output = mcp_execute(tool_name, tool_args)

        # Return result in ToolMessage format
        results.append(
            ToolMessage(
                content=str(tool_output),
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": results}


# ──────────────────────────────────────────────────────────────
# 9. ROUTING LOGIC
# ──────────────────────────────────────────────────────────────
def route_next_step(state: AgentState) -> Literal["tools", "__end__"]:
    """
    If the last AI message contains tool calls, go to tools.
    Otherwise, finish.
    """
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    return END


# ──────────────────────────────────────────────────────────────
# 10. BUILD GRAPH
# ──────────────────────────────────────────────────────────────
workflow = StateGraph(AgentState)

workflow.add_node("agent", support_agent_node)
workflow.add_node("tools", tool_execution_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", route_next_step)
workflow.add_edge("tools", "agent")

app = workflow.compile()


# ──────────────────────────────────────────────────────────────
# 11. RUN TEST QUERIES
# ──────────────────────────────────────────────────────────────
queries = [
    "Hi, where is my order ORD123?",
    "Is ORD321 delivered yet?",
    "I bought something 45 days ago, can I get refund?",
    "Check ORD999 and also tell refund for 10 days purchase",
    "What about order ORD888?",
    "Where is my order?",   # Missing order ID
]

print("\n--- MCP-Based Support Bot Running ---")

for query in queries:
    print(f"\nUser: {query}")

    result = app.invoke({
        "messages": [HumanMessage(content=query)]
    })

    final_message = result["messages"][-1].content
    print(f"Bot: {final_message}")